# OECD Data: Savings Glut Analysis

Bond yields, investment composition, and land values from OECD data.
Companion to the World Bank current account notebook.

In [1]:
# system imports
from io import StringIO

# analytic imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# local imports
import common
import mgplot as mg

In [2]:
# chart setup
CHART_DIR = "./CHARTS/Savings-Glut-OECD/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False
OECD_SOURCE = "OECD"

## 10-year government bond yields (OECD)

Long-term bond yields are more relevant to the savings glut thesis than policy
rates — the argument is that excess savings from surplus nations flowed into
government bonds, compressing yields on safe assets.

In [3]:
# OECD 10-year government bond yields (monthly)
# Dataflow: OECD.SDD.STES, DSD_STES@DF_FINMARK, 4.0
# Filter: {countries}.M.IRLT.PA._Z._Z._Z._Z.N

OECD_BOND_COUNTRIES = [
    "USA", "GBR", "JPN", "CAN", "AUS", "SWE", "CHE",
    "FRA", "ITA", "NLD", "BEL", "KOR", "EA19",
]

OECD_BOND_NAMES = {
    "USA": "United States", "GBR": "United Kingdom", "JPN": "Japan",
    "CAN": "Canada", "AUS": "Australia", "SWE": "Sweden",
    "CHE": "Switzerland", "FRA": "France",
    "ITA": "Italy", "NLD": "Netherlands", "BEL": "Belgium",
    "KOR": "Korea", "EA19": "Euro Area",
}

country_filter = "+".join(OECD_BOND_COUNTRIES)
url = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"OECD.SDD.STES,DSD_STES@DF_FINMARK,4.0/"
    f"{country_filter}.M.IRLT.PA._Z._Z._Z._Z.N"
    f"?startPeriod=1980-01&format=csv"
)
raw = common.request_get(url)
bond_df = pd.read_csv(StringIO(raw.decode("utf-8")))
bonds = bond_df.pivot(index="TIME_PERIOD", columns="REF_AREA", values="OBS_VALUE")
bonds.index = pd.PeriodIndex(bonds.index, freq="M")
bonds = bonds.sort_index().rename(columns=OECD_BOND_NAMES)
bonds = bonds.dropna(how="all", axis=0).dropna(how="all", axis=1)

print(f"Shape: {bonds.shape}")
print(f"Range: {bonds.index.min()} to {bonds.index.max()}")
print(f"Nations: {list(bonds.columns)}")
bonds.tail()

Shape: (554, 13)
Range: 1980-01 to 2026-02
Nations: ['Australia', 'Belgium', 'Canada', 'Switzerland', 'Euro Area', 'France', 'United Kingdom', 'Italy', 'Japan', 'Korea', 'Netherlands', 'Sweden', 'United States']


REF_AREA,Australia,Belgium,Canada,Switzerland,Euro Area,France,United Kingdom,Italy,Japan,Korea,Netherlands,Sweden,United States
TIME_PERIOD,,,,,,,,,,,,,
2025-10,4.234,3.19,3.128182,0.18,3.123922,3.44,4.5721,3.442,1.655,2.933,2.794,2.57683,4.06
2025-11,4.416,3.21,3.184286,0.19,3.136158,3.44,4.4985,3.459,1.805,3.248,2.831,2.65790,4.09
2025-12,4.719,3.32,3.422667,0.33,3.244733,3.56,4.4826,3.552,2.060,3.366,2.968,2.82168,4.14
2026-01,4.750,3.36,3.402381,0.27,3.220984,3.53,4.4510,3.492,2.240,3.485,2.940,2.80040,4.21
2026-02,4.770,3.30,3.288421,0.25,NaN,3.40,4.4324,3.388,2.110,3.612,2.849,2.64000,4.13


In [4]:
# Key economies bond yields
key_bonds = ["United States", "Euro Area", "Japan", "United Kingdom", "Australia", "Canada"]
available_bonds = [k for k in key_bonds if k in bonds.columns]

mg.line_plot_finalise(
    bonds[available_bonds].dropna(how="all"),
    title="10-year government bond yields since 1980",
    ylabel="Per cent per annum",
    width=2,
    y0=True,
    zero_y=True,
    rfooter="OECD",
    lfooter="Monthly data. 10-year government bond yields.",
    legend={"loc": "best", "fontsize": "x-small", "ncol": 2},
    show=SHOW,
)

In [5]:
# All available nations
mg.line_plot_finalise(
    bonds.dropna(how="all"),
    title="10-year government bond yields - all available nations",
    ylabel="Per cent per annum",
    width=1.5,
    y0=True,
    zero_y=True,
    rfooter="OECD",
    lfooter="Monthly data. 10-year government bond yields.",
    legend={"loc": "best", "fontsize": "xx-small", "ncol": 3},
    show=SHOW,
)

## Investment composition: dwellings vs productive assets (OECD)

If capital flowed from surplus nations into deficit nations' housing markets
rather than productive investment, we should see dwellings as a share of total
GFCF rising in the deficit economies. The OECD publishes annual GFCF by asset
type for member nations.

In [6]:
# Fetch OECD GFCF by asset type — annual, current prices, national currency
# Dataflow: OECD.SDD.NAD, DSD_NAMAIN10@DF_TABLE1_EXPENDITURE_GFCF_ASSET, 2.0
# Dims: FREQ.REF_AREA.SECTOR.COUNTERPART_SECTOR.TRANSACTION.INSTR_ASSET.
#       ACTIVITY.EXPENDITURE.UNIT_MEASURE.PRICE_BASE.TRANSFORMATION.TABLE_IDENTIFIER

OECD_SOURCE = "OECD"

# Deficit economies + key surplus economies for contrast
GFCF_COUNTRIES = {
    # Deficit nations
    "USA": "United States", "GBR": "United Kingdom",
    "AUS": "Australia", "CAN": "Canada",
    # Surplus nations for contrast
    "DEU": "Germany", "JPN": "Japan", "KOR": "Korea",
}

country_filter = "+".join(GFCF_COUNTRIES.keys())
asset_filter = "N11G+N111G"  # total GFCF + dwellings

url = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"OECD.SDD.NAD,DSD_NAMAIN10@DF_TABLE1_EXPENDITURE_GFCF_ASSET,2.0/"
    f"A.{country_filter}.S1..P51G.{asset_filter}._T._Z.XDC.V.N.T0102"
    f"?startPeriod=1980&format=csv"
)

print("Fetching OECD GFCF by asset type...")
raw = common.request_get(url)
gfcf_df = pd.read_csv(StringIO(raw.decode("utf-8")))
print(f"Rows: {len(gfcf_df)}")
print(f"Columns: {list(gfcf_df.columns)}")
print(f"Asset codes: {gfcf_df['INSTR_ASSET'].unique()}")
print(f"Countries: {gfcf_df['REF_AREA'].unique()}")
print(f"\nSample data:")
display(gfcf_df[["REF_AREA", "INSTR_ASSET", "TIME_PERIOD", "OBS_VALUE"]].head(20))

Fetching OECD GFCF by asset type...
Rows: 635
Columns: ['DATAFLOW', 'FREQ', 'REF_AREA', 'SECTOR', 'COUNTERPART_SECTOR', 'TRANSACTION', 'INSTR_ASSET', 'ACTIVITY', 'EXPENDITURE', 'UNIT_MEASURE', 'PRICE_BASE', 'TRANSFORMATION', 'TABLE_IDENTIFIER', 'TIME_PERIOD', 'OBS_VALUE', 'REF_YEAR_PRICE', 'CONF_STATUS', 'DECIMALS', 'OBS_STATUS', 'UNIT_MULT', 'CURRENCY']
Asset codes: <ArrowStringArray>
['N111G', 'N11G']
Length: 2, dtype: str
Countries: <ArrowStringArray>
['DEU', 'GBR', 'JPN', 'KOR', 'USA', 'CAN', 'AUS']
Length: 7, dtype: str

Sample data:


,REF_AREA,INSTR_ASSET,TIME_PERIOD,OBS_VALUE
0,DEU,N111G,1980,59583.596
1,DEU,N111G,1981,59819.504
2,DEU,N111G,1982,58898.340
3,DEU,N111G,1983,63638.965
4,DEU,N111G,1984,66705.768
5,DEU,N111G,1985,61538.262
6,DEU,N111G,1986,62223.518
7,DEU,N111G,1987,63167.150
8,DEU,N111G,1988,67447.193
9,DEU,N111G,1989,74490.729


In [7]:
# Calculate dwellings as % of total GFCF for each country
dwellings_pct = pd.DataFrame()

for code, name in GFCF_COUNTRIES.items():
    country_data = gfcf_df[gfcf_df["REF_AREA"] == code]

    total = (
        country_data[country_data["INSTR_ASSET"] == "N11G"]
        .groupby("TIME_PERIOD")["OBS_VALUE"].sum()
    )
    dwell = (
        country_data[country_data["INSTR_ASSET"] == "N111G"]
        .groupby("TIME_PERIOD")["OBS_VALUE"].sum()
    )

    pct = (dwell / total * 100).dropna()
    if pct.empty:
        print(f"  {name}: no data")
        continue
    pct.index = pd.PeriodIndex(pct.index.astype(str), freq="Y")
    dwellings_pct[name] = pct
    print(f"  {name}: {pct.index.min()} to {pct.index.max()}, latest = {pct.iloc[-1]:.1f}%")

dwellings_pct = dwellings_pct.sort_index().dropna(how="all")
print(f"\nRange: {dwellings_pct.index.min()} to {dwellings_pct.index.max()}")
dwellings_pct.tail()

  United States: 1980 to 2024, latest = 18.7%
  United Kingdom: 1980 to 2025, latest = 20.5%
  Australia: 1980 to 2024, latest = 22.3%
  Canada: 1980 to 2025, latest = 33.4%
  Germany: 1980 to 2025, latest = 30.1%
  Japan: 1980 to 2023, latest = 14.9%
  Korea: 1980 to 2024, latest = 15.8%

Range: 1980 to 2024


,United States,United Kingdom,Australia,Canada,Germany,Japan,Korea
TIME_PERIOD,,,,,,,
2020,19.771659,19.569221,22.133043,36.418343,32.159636,15.013059,17.646272
2021,22.178472,21.155941,22.103293,40.400296,32.224751,15.266113,17.861695
2022,20.967169,22.280400,22.183295,37.780961,32.170467,15.286674,16.820966
2023,18.533979,21.051701,21.461394,34.262450,31.038548,14.931402,16.771097
2024,18.728387,20.697578,22.296064,33.868549,30.538059,NaN,15.831493


In [8]:
# Deficit nations: dwellings as % of total GFCF
deficit_gfcf = ["United States", "United Kingdom", "Australia", "Canada"]
available_deficit = [c for c in deficit_gfcf if c in dwellings_pct.columns]

mg.line_plot_finalise(
    dwellings_pct[available_deficit].dropna(how="all"),
    title="Dwellings as share of total investment - deficit nations",
    ylabel="Per cent of total GFCF",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Dwellings GFCF as percentage of total GFCF. Current prices, national currency.",
    legend={"loc": "best", "fontsize": "small"},
    show=SHOW,
)

In [9]:
# Surplus nations: dwellings as % of total GFCF for contrast
surplus_gfcf = ["Germany", "Japan", "Korea"]
available_surplus = [c for c in surplus_gfcf if c in dwellings_pct.columns]

mg.line_plot_finalise(
    dwellings_pct[available_surplus].dropna(how="all"),
    title="Dwellings as share of total investment - surplus nations",
    ylabel="Per cent of total GFCF",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Dwellings GFCF as percentage of total GFCF. Current prices, national currency.",
    legend={"loc": "best", "fontsize": "small"},
    show=SHOW,
)

In [10]:
# All nations on one chart for direct comparison
mg.line_plot_finalise(
    dwellings_pct.dropna(how="all"),
    title="Dwellings as share of total investment - all nations",
    ylabel="Per cent of total GFCF",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Dwellings GFCF as percentage of total GFCF. Current prices, national currency.",
    legend={"loc": "best", "fontsize": "x-small", "ncol": 2},
    show=SHOW,
)

## Land values as share of GDP (OECD non-financial balance sheets)

GFCF excludes land (a non-produced asset in SNA accounting). Yet land is a
real cost of investment — you cannot build a factory or house without it.
If surplus-nation capital flowed into deficit-nation land markets, it would
not appear in GFCF at all. The OECD non-financial balance sheets (Table 9B)
include land values (AN.211), allowing us to track land as a share of GDP.

Note: USA does not report land values to the OECD.

In [11]:
# Fetch land values from OECD non-financial balance sheets (Table 9B)
# Asset code N211N = land (net)
# Total economy (S1), annual, current prices, national currency

LAND_COUNTRIES = {
    "GBR": "United Kingdom", "AUS": "Australia", "CAN": "Canada",
    "DEU": "Germany", "JPN": "Japan", "KOR": "Korea",
}

import time

land_values = {}
for c, name in LAND_COUNTRIES.items():
    url = (
        "https://sdmx.oecd.org/public/rest/data/"
        "OECD.SDD.NAD,DSD_NASEC10@DF_TABLE9B,1.1/"
        f"A.{c}.S1.S1.A._Z.N211N._Z.XDC._Z.V.N.T2600"
        "?startPeriod=1990&format=csv"
    )
    try:
        raw = common.request_get(url)
        df = pd.read_csv(StringIO(raw.decode("utf-8")))
        mult = int(df["UNIT_MULT"].iloc[0])
        s = df.set_index("TIME_PERIOD")["OBS_VALUE"] * (10 ** mult)
        land_values[name] = s
        print(f"  {name}: {s.index.min()} to {s.index.max()}")
    except Exception as e:
        print(f"  {name}: failed ({e})")
    time.sleep(1)  # respect OECD rate limits

print(f"\nLand data for {len(land_values)} countries")

  United Kingdom: 1995 to 2023
  Australia: 1990 to 2024
  Canada: 1990 to 2024
  Germany: 1999 to 2024
  Japan: 1994 to 2023
  Korea: 1995 to 2024

Land data for 6 countries


In [12]:
# Fetch nominal GDP (current prices, national currency) from OECD Table 1
# Transaction B1GQ = GDP

gdp_values = {}
for c, name in LAND_COUNTRIES.items():
    url = (
        "https://sdmx.oecd.org/public/rest/data/"
        "OECD.SDD.NAD,DSD_NAMAIN10@DF_TABLE1,2.0/"
        f"A.{c}.S1.S1.B1GQ._Z._Z._Z.XDC.V.N.T0101"
        "?startPeriod=1990&format=csv"
    )
    try:
        raw = common.request_get(url)
        df = pd.read_csv(StringIO(raw.decode("utf-8")))
        mult = int(df["UNIT_MULT"].iloc[0])
        s = df.set_index("TIME_PERIOD")["OBS_VALUE"] * (10 ** mult)
        gdp_values[name] = s
        print(f"  {name}: {s.index.min()} to {s.index.max()}")
    except Exception as e:
        print(f"  {name}: failed ({e})")
    time.sleep(1)

print(f"\nGDP data for {len(gdp_values)} countries")

  United Kingdom: 1990 to 2025
  Australia: 1990 to 2024
  Canada: 1990 to 2025
  Germany: 1990 to 2025
  Japan: 1990 to 2023
  Korea: 1990 to 2024

GDP data for 6 countries


In [13]:
# Calculate land value as % of GDP
land_gdp = pd.DataFrame()
for name in LAND_COUNTRIES.values():
    if name in land_values and name in gdp_values:
        land = land_values[name]
        gdp = gdp_values[name]
        common_idx = land.index.intersection(gdp.index)
        ratio = (land[common_idx] / gdp[common_idx] * 100)
        land_gdp[name] = ratio
        print(f"  {name}: latest = {ratio.iloc[-1]:.1f}% of GDP")

land_gdp.index = pd.PeriodIndex(land_gdp.index.astype(str), freq="Y")
land_gdp = land_gdp.sort_index()
land_gdp.tail()

  United Kingdom: latest = 130.9% of GDP
  Australia: latest = 300.5% of GDP
  Canada: latest = 101.4% of GDP
  Germany: latest = 113.2% of GDP
  Japan: latest = 323.8% of GDP
  Korea: latest = 359.6% of GDP


,United Kingdom,Australia,Canada,Germany,Japan,Korea
TIME_PERIOD,,,,,,
2019,258.035758,321.899481,197.207344,154.216997,223.142803,446.222985
2020,294.530291,371.621929,244.617701,176.802291,234.329560,503.678671
2021,302.793100,383.785884,280.905453,178.879924,231.280404,552.152859
2022,279.160030,349.563623,208.492807,177.180546,232.547966,517.142098
2023,243.353448,364.787642,209.877981,164.140037,227.758309,493.625487


In [14]:
# Deficit nations: land value as % of GDP
deficit_land = ["United Kingdom", "Australia", "Canada"]
available = [c for c in deficit_land if c in land_gdp.columns]

mg.line_plot_finalise(
    land_gdp[available].dropna(how="all"),
    title="Land values as share of GDP - deficit nations",
    ylabel="Per cent of GDP",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Non-financial balance sheet land values (AN.211) / nominal GDP. USA not available.",
    legend={"loc": "best", "fontsize": "small"},
    show=SHOW,
)

In [15]:
# Surplus nations: land value as % of GDP
surplus_land = ["Germany", "Japan", "Korea"]
available = [c for c in surplus_land if c in land_gdp.columns]

mg.line_plot_finalise(
    land_gdp[available].dropna(how="all"),
    title="Land values as share of GDP - surplus nations",
    ylabel="Per cent of GDP",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Non-financial balance sheet land values (AN.211) / nominal GDP.",
    legend={"loc": "best", "fontsize": "small"},
    show=SHOW,
)

In [16]:
# All nations on one chart
mg.line_plot_finalise(
    land_gdp.dropna(how="all"),
    title="Land values as share of GDP",
    ylabel="Per cent of GDP",
    width=2,
    y0=True,
    rfooter=OECD_SOURCE,
    lfooter="Non-financial balance sheet land values (AN.211) / nominal GDP. USA not available.",
    legend={"loc": "best", "fontsize": "x-small", "ncol": 2},
    show=SHOW,
)

## The End

In [17]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-03-14 10:14:12

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.10.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

matplotlib: 3.10.8
mgplot    : 0.2.19
numpy     : 2.4.2
pandas    : 3.0.1

Watermark: 2.6.0

